# 🤖 Laptop Price Prediction — Model Development & Evaluation

**Notebook:** `02_modeling.ipynb`  
**Depends on:** `src/preprocessing.py`, `src/features.py`  
**Goal:** Train, compare, tune, and evaluate regression models for laptop price prediction.

---
> **Rules:** Test set is touched exactly once — final evaluation only. All tuning uses CV on train set.

---
## 1. Setup & Data Loading

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_validate, KFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import shap

from src.preprocessing import load_and_clean
from src.features import engineer_features

# ── Plot style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120, "figure.facecolor": "#0f1117",
    "axes.facecolor": "#1a1d27", "axes.edgecolor": "#3a3f5c",
    "axes.labelcolor": "#c8cce0", "axes.titlesize": 12,
    "axes.titlecolor": "#e8eaf6", "axes.titleweight": "bold",
    "xtick.color": "#9199c2", "ytick.color": "#9199c2",
    "text.color": "#c8cce0", "grid.color": "#2c3057",
    "grid.linewidth": 0.6, "legend.facecolor": "#1a1d27",
    "legend.edgecolor": "#3a3f5c", "legend.labelcolor": "#c8cce0",
})
ACCENT, HIGHLIGHT = "#7c83fd", "#fd7c83"

SEED = 42
np.random.seed(SEED)

def fmt_inr(x, pos=None): return f"₹{x/1000:.0f}K"

print("Setup complete.")

In [ ]:
# Load and engineer features
df_raw = load_and_clean("../laptop data/laptop_data.csv")
df     = engineer_features(df_raw)
print(f"Dataset: {df.shape[0]} rows × {df.shape[1]} columns")

---
## 2. Feature Selection & Train/Test Split

In [ ]:
# ── Feature groups ─────────────────────────────────────────────────────────
# total_pixels chosen over width/height to avoid collinearity
NUM_FEATURES = [
    "Inches", "ram_gb", "weight_kg", "total_pixels",
    "is_touchscreen", "is_ips", "cpu_speed_ghz",
    "ssd_gb", "hdd_gb", "flash_storage_gb", "total_storage_gb",
]
OHE_FEATURES = [
    "Company", "TypeName", "OpSys", "cpu_brand", "gpu_brand",
    "primary_storage_type",
]
# Ordinal encoding — ordered from cheapest to most expensive tier
CPU_ORDER = [["Atom", "AMD E-Series", "Celeron", "Pentium", "AMD A-Series",
              "ARM Cortex", "Core i3", "Core M", "Ryzen",
              "Core i5", "Core i7", "Xeon", "Other"]]
GPU_ORDER = [["Intel HD", "Intel UHD", "Intel Iris", "Intel Iris Plus",
              "Intel Iris Pro", "AMD Radeon R", "AMD Radeon", "AMD Radeon Pro",
              "AMD FirePro", "ARM Mali", "GeForce", "GeForce GTX",
              "GeForce RTX", "Quadro", "Other GPU"]]
ORD_CPU = ["cpu_family"]
ORD_GPU = ["gpu_family"]

ALL_FEATURES = NUM_FEATURES + OHE_FEATURES + ORD_CPU + ORD_GPU
TARGET = "Price"

X = df[ALL_FEATURES]
y = df[TARGET]

# ── 80/20 split ────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED)

y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

print(f"Train : {X_train.shape[0]} rows")
print(f"Test  : {X_test.shape[0]} rows  (HELD OUT — not touched until final eval)")
print(f"\nFeatures: {len(ALL_FEATURES)} total")
print(f"  Numerical  : {len(NUM_FEATURES)}")
print(f"  OHE cats   : {len(OHE_FEATURES)}")
print(f"  Ordinal    : {len(ORD_CPU)+len(ORD_GPU)}")

---
## 3. Preprocessing Pipeline

In [ ]:
def build_preprocessor():
    """Build a fresh ColumnTransformer (unfitted)."""
    return ColumnTransformer(transformers=[
        ("num", StandardScaler(), NUM_FEATURES),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), OHE_FEATURES),
        ("ord_cpu", OrdinalEncoder(categories=CPU_ORDER,
                                   handle_unknown="use_encoded_value", unknown_value=-1), ORD_CPU),
        ("ord_gpu", OrdinalEncoder(categories=GPU_ORDER,
                                   handle_unknown="use_encoded_value", unknown_value=-1), ORD_GPU),
    ], remainder="drop")

def make_pipeline(estimator):
    """Wrap an estimator in a full preprocessing pipeline."""
    return Pipeline([("prep", build_preprocessor()), ("model", estimator)])

print("Preprocessor factory ready.")
print("Fitted only on training folds — no leakage.")

---
## 4. Cross-Validation Utilities

In [ ]:
CV = KFold(n_splits=5, shuffle=True, random_state=SEED)

def run_cv(pipeline, X_tr, y_tr, is_log=False):
    """
    Run 5-fold CV. For log-price models, inverse-transform before
    computing MAE/RMSE so metrics are always in INR.
    Returns dict with cv_r2, cv_mae, cv_rmse (all in INR).
    """
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    r2s, maes, rmses = [], [], []

    for tr_idx, val_idx in kf.split(X_tr):
        X_f, X_v = X_tr.iloc[tr_idx], X_tr.iloc[val_idx]
        y_f, y_v = y_tr.iloc[tr_idx], y_tr.iloc[val_idx]

        pipeline.fit(X_f, y_f)
        raw_preds = pipeline.predict(X_v)

        if is_log:
            preds_inr = np.expm1(raw_preds)
            y_inr     = np.expm1(y_v)
        else:
            preds_inr = raw_preds
            y_inr     = y_v

        r2s.append(r2_score(y_inr, preds_inr))
        maes.append(mean_absolute_error(y_inr, preds_inr))
        rmses.append(np.sqrt(mean_squared_error(y_inr, preds_inr)))

    return {
        "cv_r2":   np.mean(r2s),
        "cv_mae":  np.mean(maes),
        "cv_rmse": np.mean(rmses),
    }


def eval_on_test(pipeline, X_tr, y_tr_fit, X_te, y_te_inr, is_log=False):
    """
    Refit on full training set, evaluate once on test set.
    Returns dict with test_r2, test_mae, test_rmse in INR.
    """
    pipeline.fit(X_tr, y_tr_fit)
    raw_preds = pipeline.predict(X_te)
    preds_inr = np.expm1(raw_preds) if is_log else raw_preds

    return {
        "test_r2":   r2_score(y_te_inr, preds_inr),
        "test_mae":  mean_absolute_error(y_te_inr, preds_inr),
        "test_rmse": np.sqrt(mean_squared_error(y_te_inr, preds_inr)),
        "preds_inr": preds_inr,
    }

print("CV and test evaluation utilities ready.")

---
## 5. Baseline & Linear Models

Both raw-price and log-price variants are evaluated.

In [ ]:
results = []   # accumulates all model results

# Helper to register a result row
def add_result(name, target_str, cv_metrics, test_metrics):
    results.append({
        "Model": name,
        "Target": target_str,
        "CV R²":   round(cv_metrics["cv_r2"],   4),
        "CV MAE":  round(cv_metrics["cv_mae"],   0),
        "CV RMSE": round(cv_metrics["cv_rmse"],  0),
        "Test R²":   round(test_metrics["test_r2"],   4),
        "Test MAE":  round(test_metrics["test_mae"],  0),
        "Test RMSE": round(test_metrics["test_rmse"], 0),
        "_preds":  test_metrics["preds_inr"],
    })
    print(f"  [{name} | {target_str}]  CV R²={cv_metrics['cv_r2']:.4f}  "
          f"CV RMSE=₹{cv_metrics['cv_rmse']:,.0f}  "
          f"Test R²={test_metrics['test_r2']:.4f}  "
          f"Test RMSE=₹{test_metrics['test_rmse']:,.0f}")

print("=" * 72)
print(" BASELINE & LINEAR MODELS")
print("=" * 72)

for name, estimator in [
    ("DummyRegressor",   DummyRegressor(strategy="mean")),
    ("LinearRegression", LinearRegression()),
    ("Ridge",            Ridge(alpha=10.0)),
]:
    for is_log, tag, y_tr_fit in [
        (False, "Raw",  y_train),
        (True,  "Log",  y_train_log),
    ]:
        import copy
        pipe = make_pipeline(copy.deepcopy(estimator))
        cv   = run_cv(pipe, X_train, y_tr_fit, is_log=is_log)
        te   = eval_on_test(pipe, X_train, y_tr_fit, X_test, y_test, is_log=is_log)
        add_result(name, tag, cv, te)

---
## 6. Tree-Based Models

In [ ]:
print("=" * 72)
print(" TREE-BASED MODELS (default hyperparameters)")
print("=" * 72)

tree_models = [
    ("RandomForest",        RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1)),
    ("GradientBoosting",    GradientBoostingRegressor(n_estimators=200, random_state=SEED)),
    ("XGBoost",             XGBRegressor(n_estimators=300, learning_rate=0.1, random_state=SEED,
                                         verbosity=0, n_jobs=-1)),
]

for name, estimator in tree_models:
    import copy
    for is_log, tag, y_tr_fit in [
        (False, "Raw", y_train),
        (True,  "Log", y_train_log),
    ]:
        pipe = make_pipeline(copy.deepcopy(estimator))
        cv   = run_cv(pipe, X_train, y_tr_fit, is_log=is_log)
        te   = eval_on_test(pipe, X_train, y_tr_fit, X_test, y_test, is_log=is_log)
        add_result(name, tag, cv, te)

---
## 7. Cross-Validation Comparison Table

In [ ]:
results_df = pd.DataFrame(results).drop(columns=["_preds"])
results_df = results_df.sort_values("CV RMSE")

# Format currency columns
display_df = results_df.copy()
for col in ["CV MAE", "CV RMSE", "Test MAE", "Test RMSE"]:
    display_df[col] = display_df[col].apply(lambda x: f"₹{x:,.0f}")

print(display_df.to_string(index=False))

---
## 8. Hyperparameter Tuning — Top 2 Models

Using the **log-price** target for the best-performing model configuration identified above.

In [ ]:
print("Tuning XGBoost (log-price target)...")

xgb_param_grid = {
    "model__n_estimators":     [200, 400, 600],
    "model__max_depth":        [3, 4, 5, 6],
    "model__learning_rate":    [0.05, 0.08, 0.10, 0.15],
    "model__subsample":        [0.7, 0.8, 0.9],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
}

xgb_base = make_pipeline(XGBRegressor(random_state=SEED, verbosity=0, n_jobs=-1))

xgb_search = RandomizedSearchCV(
    xgb_base,
    param_distributions=xgb_param_grid,
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=CV,
    random_state=SEED,
    n_jobs=-1,
    refit=True,
    verbose=0,
)
xgb_search.fit(X_train, y_train_log)

print(f"\nBest XGBoost params:")
for k, v in xgb_search.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nBest CV RMSE (log scale): {-xgb_search.best_score_:.6f}")

In [ ]:
print("Tuning RandomForest (log-price target)...")

rf_param_grid = {
    "model__n_estimators":  [200, 300, 500],
    "model__max_depth":     [None, 15, 20, 25],
    "model__max_features":  ["sqrt", "log2", 0.5, 0.7],
    "model__min_samples_leaf": [1, 2, 4],
}

rf_base = make_pipeline(RandomForestRegressor(random_state=SEED, n_jobs=-1))

rf_search = RandomizedSearchCV(
    rf_base,
    param_distributions=rf_param_grid,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=CV,
    random_state=SEED,
    n_jobs=-1,
    refit=True,
    verbose=0,
)
rf_search.fit(X_train, y_train_log)

print(f"\nBest RandomForest params:")
for k, v in rf_search.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nBest CV RMSE (log scale): {-rf_search.best_score_:.6f}")

In [ ]:
# Evaluate tuned models — CV in INR, then final test evaluation
print("=" * 72)
print(" TUNED MODEL EVALUATION")
print("=" * 72)

tuned_models = [
    ("XGBoost (Tuned)", xgb_search.best_estimator_),
    ("RandomForest (Tuned)", rf_search.best_estimator_),
]

import copy
for name, best_pipe in tuned_models:
    # Re-run proper INR CV (search used log-scale scoring)
    pipe_copy = copy.deepcopy(best_pipe)
    cv   = run_cv(pipe_copy, X_train, y_train_log, is_log=True)
    te   = eval_on_test(pipe_copy, X_train, y_train_log, X_test, y_test, is_log=True)
    add_result(name, "Log", cv, te)

# Refresh full results table
results_df = pd.DataFrame(results).drop(columns=["_preds"])
results_df = results_df.sort_values("CV RMSE")
display_df = results_df.copy()
for col in ["CV MAE", "CV RMSE", "Test MAE", "Test RMSE"]:
    display_df[col] = display_df[col].apply(lambda x: f"₹{x:,.0f}")

print("\n" + display_df.to_string(index=False))

---
## 9. Model Comparison Chart

In [ ]:
plot_df = pd.DataFrame(results).drop(columns=["_preds"]).sort_values("CV RMSE")
labels  = plot_df["Model"] + "\n(" + plot_df["Target"] + ")"

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Model Comparison — CV Performance (INR)",
             fontsize=13, color="#e8eaf6", fontweight="bold")

metrics = [("CV R²", "R²"), ("CV MAE", "MAE (₹)"), ("CV RMSE", "RMSE (₹)")]
palettes = [plt.cm.plasma(np.linspace(0.2, 0.85, len(plot_df)))] * 3

for ax, (col, ylabel), colors in zip(axes, metrics, palettes):
    vals   = plot_df[col].values
    # Sort ascending for RMSE/MAE, descending for R²
    order  = np.argsort(vals) if col != "CV R²" else np.argsort(vals)[::-1]
    sorted_labels = [labels.values[i] for i in order]
    sorted_vals   = vals[order]
    bar_colors    = plt.cm.plasma(np.linspace(0.2, 0.85, len(sorted_vals)))

    bars = ax.barh(sorted_labels, sorted_vals, color=bar_colors, edgecolor="none", alpha=0.85)
    ax.set_title(ylabel)
    ax.grid(axis="x", alpha=0.3)
    for bar, val in zip(bars, sorted_vals):
        if col == "CV R²":
            label_str = f"{val:.3f}"
        else:
            label_str = f"₹{val:,.0f}"
        ax.text(val * 1.01, bar.get_y() + bar.get_height() / 2,
                label_str, va="center", fontsize=6.5, color="#9199c2")

plt.tight_layout()
plt.show()

---
## 10. Best Model — Final Test Evaluation

In [ ]:
# Select best model by CV RMSE
best_row   = pd.DataFrame(results).sort_values("cv_rmse").iloc[0]
best_name  = best_row["Model"]
best_preds = best_row["_preds"]

print("=" * 60)
print(f"  BEST MODEL : {best_name} ({best_row['Target']} target)")
print("=" * 60)
print(f"  CV  R²   : {best_row['cv_r2']:.4f}")
print(f"  CV  MAE  : ₹{best_row['cv_mae']:,.0f}")
print(f"  CV  RMSE : ₹{best_row['cv_rmse']:,.0f}")
print("-" * 60)
print(f"  Test R²   : {best_row['test_r2']:.4f}")
print(f"  Test MAE  : ₹{best_row['test_mae']:,.0f}")
print(f"  Test RMSE : ₹{best_row['test_rmse']:,.0f}")
print("=" * 60)

---
## 11. Regression Diagnostics

In [ ]:
y_true = y_test.values
y_pred = best_preds
residuals = y_true - y_pred
pct_errors = (residuals / y_true) * 100

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f"Regression Diagnostics — {best_name}",
             fontsize=13, color="#e8eaf6", fontweight="bold")

# 1. Actual vs Predicted
ax = axes[0, 0]
ax.scatter(y_true / 1000, y_pred / 1000, alpha=0.4, s=18,
           color=ACCENT, edgecolors="none")
mn, mx = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
ax.plot([mn/1000, mx/1000], [mn/1000, mx/1000],
        color=HIGHLIGHT, linewidth=1.5, linestyle="--", label="Perfect")
ax.set_title("Actual vs Predicted Price")
ax.set_xlabel("Actual (₹K)")
ax.set_ylabel("Predicted (₹K)")
ax.legend()
ax.grid(alpha=0.3)

# 2. Residuals vs Predicted
ax = axes[0, 1]
ax.scatter(y_pred / 1000, residuals / 1000, alpha=0.4, s=18,
           color=ACCENT, edgecolors="none")
ax.axhline(0, color=HIGHLIGHT, linewidth=1.5, linestyle="--")
ax.set_title("Residuals vs Predicted")
ax.set_xlabel("Predicted (₹K)")
ax.set_ylabel("Residual (₹K)")
ax.grid(alpha=0.3)

# 3. Residual distribution
ax = axes[1, 0]
ax.hist(residuals / 1000, bins=40, color=ACCENT, edgecolor="none", alpha=0.8)
ax.axvline(0, color=HIGHLIGHT, linewidth=1.5, linestyle="--")
ax.axvline(np.mean(residuals)/1000, color="#ffd700", linewidth=1.2,
           linestyle="-.", label=f"Mean ₹{np.mean(residuals)/1000:.1f}K")
ax.set_title("Residual Distribution")
ax.set_xlabel("Residual (₹K)")
ax.set_ylabel("Count")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)

# 4. % Error distribution
ax = axes[1, 1]
ax.hist(pct_errors, bins=40, color=HIGHLIGHT, edgecolor="none", alpha=0.8)
ax.axvline(0, color=ACCENT, linewidth=1.5, linestyle="--")
within_10  = (np.abs(pct_errors) <= 10).mean() * 100
within_20  = (np.abs(pct_errors) <= 20).mean() * 100
ax.set_title("% Prediction Error Distribution")
ax.set_xlabel("% Error  ((Actual−Pred)/Actual × 100)")
ax.set_ylabel("Count")
ax.text(0.97, 0.95, f"Within ±10%: {within_10:.1f}%\nWithin ±20%: {within_20:.1f}%",
        transform=ax.transAxes, ha="right", va="top",
        fontsize=9, color="#c8cce0",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#1a1d27", edgecolor="#3a3f5c"))
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Predictions within ±10% of actual: {within_10:.1f}%")
print(f"Predictions within ±20% of actual: {within_20:.1f}%")
print(f"Residual skewness: {pd.Series(residuals).skew():.3f}")

---
## 12. Feature Importance

In [ ]:
# Retrieve feature importances from the best tree-based fitted pipeline
# Use the last-fitted pipeline for the best model
all_results_with_preds = pd.DataFrame(results).sort_values("cv_rmse")
best_idx = all_results_with_preds.index[0]

# Find the best tuned tree model — re-fit it cleanly to get the pipeline object
# Determine which tuned model won
best_model_name = all_results_with_preds.iloc[0]["Model"]
is_log_best     = all_results_with_preds.iloc[0]["Target"] == "Log"
y_tr_best       = y_train_log if is_log_best else y_train

if "XGBoost" in best_model_name and "Tuned" in best_model_name:
    best_fitted_pipe = copy.deepcopy(xgb_search.best_estimator_)
elif "RandomForest" in best_model_name and "Tuned" in best_model_name:
    best_fitted_pipe = copy.deepcopy(rf_search.best_estimator_)
else:
    # Fall back — re-make with defaults
    if "XGBoost" in best_model_name:
        est = XGBRegressor(n_estimators=300, learning_rate=0.1, random_state=SEED, verbosity=0)
    elif "RandomForest" in best_model_name:
        est = RandomForestRegressor(n_estimators=200, random_state=SEED)
    elif "GradientBoosting" in best_model_name:
        est = GradientBoostingRegressor(n_estimators=200, random_state=SEED)
    else:
        est = Ridge()
    best_fitted_pipe = make_pipeline(est)

best_fitted_pipe.fit(X_train, y_tr_best)

# Get feature names from ColumnTransformer
prep = best_fitted_pipe.named_steps["prep"]
ohe_names = prep.named_transformers_["ohe"].get_feature_names_out(OHE_FEATURES).tolist()
feat_names = NUM_FEATURES + ohe_names + ORD_CPU + ORD_GPU

model_step = best_fitted_pipe.named_steps["model"]
importances = model_step.feature_importances_

imp_df = pd.DataFrame({"Feature": feat_names, "Importance": importances})
imp_df = imp_df.sort_values("Importance", ascending=False).head(25)

fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(imp_df)))
bars = ax.barh(imp_df["Feature"][::-1], imp_df["Importance"][::-1],
               color=colors[::-1], edgecolor="none", alpha=0.9)
ax.set_title(f"Top 25 Feature Importances — {best_model_name}",
             fontsize=12)
ax.set_xlabel("Importance Score")
ax.grid(axis="x", alpha=0.3)
for bar, val in zip(bars, imp_df["Importance"][::-1]):
    ax.text(val + imp_df["Importance"].max() * 0.005,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=7, color="#9199c2")
plt.tight_layout()
plt.show()

print("\nTop 10 features:")
print(imp_df.head(10).to_string(index=False))

---
## 13. SHAP Analysis

In [ ]:
# Transform test set through the fitted preprocessor for SHAP
X_test_transformed = prep.transform(X_test)

explainer = shap.TreeExplainer(model_step)
shap_values = explainer.shap_values(X_test_transformed)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor("#0f1117")

# Summary bar plot (mean |SHAP|)
plt.sca(axes[0])
shap.summary_plot(
    shap_values, X_test_transformed,
    feature_names=feat_names,
    plot_type="bar",
    max_display=15,
    show=False,
    color=ACCENT,
)
axes[0].set_title("SHAP — Mean |SHAP| (Top 15)", color="#e8eaf6", fontweight="bold")
axes[0].set_facecolor("#1a1d27")

# Beeswarm / dot plot
plt.sca(axes[1])
shap.summary_plot(
    shap_values, X_test_transformed,
    feature_names=feat_names,
    plot_type="dot",
    max_display=15,
    show=False,
)
axes[1].set_title("SHAP — Beeswarm (Top 15)", color="#e8eaf6", fontweight="bold")
axes[1].set_facecolor("#1a1d27")

plt.tight_layout()
plt.show()

---
## 14. Modeling Summary & Decisions

In [ ]:
# ── Auto-generate summary from actual results ──────────────────────────────
res_df    = pd.DataFrame(results).sort_values("cv_rmse")
best      = res_df.iloc[0]

# Log vs Raw — compare best of each
best_log  = res_df[res_df["Target"] == "Log"].iloc[0]
best_raw  = res_df[res_df["Target"] == "Raw"].iloc[0]
log_wins  = best_log["cv_rmse"] < best_raw["cv_rmse"]

# Overfitting: CV RMSE vs Test RMSE gap
rmse_gap  = abs(best["test_rmse"] - best["cv_rmse"])
gap_pct   = (rmse_gap / best["cv_rmse"]) * 100
overfit   = gap_pct > 15

top10_feats = imp_df.head(10)["Feature"].tolist()

print("=" * 65)
print("  MODELING SUMMARY")
print("=" * 65)
print(f"\n  Best model (by CV RMSE):")
print(f"    {best['Model']} — {best['Target']} price target")
print(f"\n  Cross-validation performance:")
print(f"    R²   = {best['cv_r2']:.4f}")
print(f"    MAE  = ₹{best['cv_mae']:,.0f}")
print(f"    RMSE = ₹{best['cv_rmse']:,.0f}")
print(f"\n  Final test set performance (evaluated ONCE):")
print(f"    R²   = {best['test_r2']:.4f}")
print(f"    MAE  = ₹{best['test_mae']:,.0f}")
print(f"    RMSE = ₹{best['test_rmse']:,.0f}")
print(f"\n  Log-price transformation better? {'YES' if log_wins else 'NO'}")
print(f"    Best log model  CV RMSE = ₹{best_log['cv_rmse']:,.0f}")
print(f"    Best raw model  CV RMSE = ₹{best_raw['cv_rmse']:,.0f}")
print(f"\n  Overfitting assessment:")
print(f"    CV→Test RMSE gap = ₹{rmse_gap:,.0f}  ({gap_pct:.1f}%)")
print(f"    {'⚠ Possible overfitting — gap > 15%' if overfit else '✓ No significant overfitting detected'}")
print(f"\n  Top 10 most important features:")
for i, feat in enumerate(top10_feats, 1):
    print(f"    {i:2}. {feat}")
print("=" * 65)

---
*End of Modeling Notebook — `02_modeling.ipynb`*